# Data Loading: Pipes to Your Model

Reach for this when you need: 
- To implement a custom data loading pipeline.
- Reference for `DataLoader` speed optimization.
- Transforming raw files into tensors efficiently.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Custom Dataset Template

| Method | Description | Usage |
| :--- | :--- | :--- |
| `__init__` | Load data paths/metadata | Preparing indices |
| `__len__` | Return total number of samples | Defining dataset size |
| `__getitem__` | Map index to sample (tensor) | Fetching/Transforming a single item |

In [ ]:
class MyDataset(Dataset):
    """
    A boilerplate for custom data loading.
    Parameters:
    - x_data (numpy array): Input features
    - y_data (numpy array): Labels
    """
    def __init__(self, x_data: np.ndarray, y_data: np.ndarray):
        self.x = torch.from_numpy(x_data).float() # Convert to tensor
        self.y = torch.from_numpy(y_data).long()
        self.n_samples = x_data.shape[0]

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.x[index], self.y[index]

    def __len__(self) -> int:
        return self.n_samples

# Usage
x = np.random.randn(100, 10)
y = np.random.randint(0, 2, size=100)
dataset = MyDataset(x, y)

## 2. DataLoader Optimization

| Param | Description | Industry Standard |
| :--- | :--- | :--- |
| `batch_size` | Number of samples per update | 16–1024 (powers of 2) |
| `num_workers` | Multiprocessing for data loading | CPU cores count or core count / 2 |
| `pin_memory` | Faster transfer to GPU | Always `True` for GPU training |
| `shuffle` | Randomize order every epoch | Use `True` for training, `False` for eval |

In [ ]:
loader = DataLoader(
    dataset=dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,    # Parallel loading
    pin_memory=True   # Speeds up to(device) calls
)

# Iteration loop
for inputs, labels in loader:
    inputs, labels = inputs.to(device), labels.to(device)
    # Training step follows

### Common Pitfalls
- **Dtype Mismatch**: Ensure `__getitem__` returns the dtype expected by the loss function (usually `float32` for data, `long` for labels).
- **CPU Bottleneck**: If `num_workers=0`, data loading happens in the main thread, often keeping the GPU idle. Always benchmark `num_workers`.
- **Transform Timing**: Do not perform expensive transforms in the main loop; move them into `__getitem__` to leverage `num_workers`.

### Key Takeaways
- `Dataset` maps indices to actual data; `DataLoader` manages batching and parallelization.
- Set `pin_memory=True` whenever training on a GPU.
- Use `num_workers > 0` to prevent data processing from slowing down training.